# 2.0 — Baseline Training: YOLOv8s (Raw, No Imbalance Handling)

Train a unified YOLOv8s model on the raw Construction Site Safety dataset **without any class imbalance handling**. This establishes the baseline performance.

**Architecture note:** All training logic lives in `algear.modeling.train`; this notebook orchestrates and visualises results.

## Setup

In [1]:
# @title Install dependencies
!pip install -q ultralytics roboflow loguru typer python-dotenv pyyaml matplotlib

In [ ]:
# @title Mount Google Drive (or clone the repo)
import os
from pathlib import Path
import sys

# Option A: Mount Drive and point to your project folder
# from google.colab import drive
# drive.mount('/content/drive')

# Update this path to match where you placed the AlGear project
# PROJECT_DIR = Path("/content/drive/MyDrive/AlGear")

# Option B: Uncomment to clone from GitHub instead
!git clone https://github.com/Hndra04/AlGear 
PROJECT_DIR = Path("/content/AlGear")

os.chdir(str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR))
print(f"Project root: {PROJECT_DIR}")
print(f"Contents: {list(PROJECT_DIR.iterdir())[:10]}")

Cloning into '/content/AlGear'...
fatal: could not read Username for 'https://github.com': No such device or address


FileNotFoundError: [Errno 2] No such file or directory: '/content/AlGear'

In [ ]:
# @title Set Roboflow API key
import os
os.environ["ROBOFLOW_API_KEY"] = ""  # <-- Paste your key here

# Verify
from algear.config import ROBOFLOW_API_KEY, ROBOFLOW_DIR, ROBOFLOW_WORKSPACE, ROBOFLOW_PROJECT, ROBOFLOW_VERSION
print(f"Workspace: {ROBOFLOW_WORKSPACE}, Project: {ROBOFLOW_PROJECT}, v{ROBOFLOW_VERSION}")
print(f"API key set: {bool(ROBOFLOW_API_KEY)}")

In [ ]:
# @title Download dataset from Roboflow (if not already present)
if not ROBOFLOW_DIR.exists():
    from algear.dataset import download_roboflow
    download_roboflow(output_dir=ROBOFLOW_DIR)
else:
    print(f"Dataset already exists at {ROBOFLOW_DIR}")

## 1. Class Distribution

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np
import yaml

from algear.config import ROBOFLOW_DIR

with open(ROBOFLOW_DIR / "data.yaml") as f:
    data_cfg = yaml.safe_load(f)
class_names = data_cfg["names"]

def count_classes(split: str) -> Counter:
    c = Counter()
    for lbl in sorted((ROBOFLOW_DIR / split / "labels").glob("*.txt")):
        with open(lbl) as f:
            for line in f:
                c[int(line.strip().split()[0])] += 1
    return c

train_counts = count_classes("train")
total = sum(train_counts.values())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
names = [class_names[i] for i in range(len(class_names))]
counts = [train_counts.get(i, 0) for i in range(len(class_names))]
colors = plt.cm.Reds(np.linspace(0.3, 1.0, len(names)))

ax1.bar(names, counts, color=colors)
ax1.set_title(f"Training Set — {total} total instances")
ax1.set_ylabel("Count")
ax1.tick_params(axis="x", rotation=30, labelsize=9)
for bar, c in zip(ax1.containers[0], counts):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
             str(c), ha="center", va="bottom", fontsize=9)

pcts = [c / total * 100 for c in counts]
ax2.bar(names, pcts, color=colors)
ax2.set_title("Class Distribution (%)")
ax2.set_ylabel("Percentage (%)")
ax2.tick_params(axis="x", rotation=30, labelsize=9)

plt.tight_layout()
plt.show()

print(f"Imbalance ratio (helmet:no-helmet):  {counts[0] / max(counts[1], 1):.1f}x")
print(f"Imbalance ratio (vest:no-vest):      {counts[4] / max(counts[2], 1):.1f}x")

## 2. Train YOLOv8s Baseline

Using `yolov8s.pt` pretrained checkpoint, 50 epochs, **no class weights** — raw imbalanced training.

In [ ]:
import torch

device = 0 if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
from algear.modeling.train import train_yolov8

results = train_yolov8(
    data_yaml=ROBOFLOW_DIR / "data.yaml",
    model_name="yolov8s.pt",
    epochs=50,
    imgsz=640,
    batch=16,
    device=device,
    output_dir=PROJECT_DIR / "models",
)

## 3. Evaluate on Test Set

In [ ]:
from algear.modeling.train import evaluate

metrics = evaluate(
    model_path=PROJECT_DIR / "models" / "baseline" / "weights" / "best.pt",
    data_yaml=ROBOFLOW_DIR / "data.yaml",
    split="test",
    device=device,
)

## 4. Baseline Results

| Metric | Value | Target |
|---|---|---|
| mAP@50 (overall) | | ≥ 80% |
| mAP@50 (no-helmet)  | | |
| mAP@50 (no-vest)    | | |
| Recall (no-helmet)  | | |
| Recall (no-vest)    | | |
| Precision (overall) | | |

**Next steps if metrics fall short:**
- Increase `cls` loss weight for minority classes
- Add focal loss (`fl_gamma=1.5`)
- Oversample minority-class images
- Train longer (100+ epochs) with early stopping
- Try YOLOv8m or data augmentation tuning